# Model Evaluation
Evaluates `best_model.pt` on both val and test splits of `sts-222`.
Architecture is **auto-detected** from the checkpoint — nothing to configure manually.

In [1]:
import sys, os, pickle
sys.path.insert(0, os.path.abspath('.'))

import torch
import pandas as pd
from models.model.transformer import Transformer
from evaluate_search import evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cpu


In [2]:
# ── Paths (edit if needed) ────────────────────────────────────────────────────
MODEL_PATH    = 'best_model.pt'
VOCAB_PATH    = 'vocab.pkl'
VAL_PATH      = 'data/sts-222/stsb_validation.csv'
TEST_PATH     = 'data/sts-222/stsb_test.csv'
POS_THRESHOLD = 0.65
BATCH_SIZE    = 64
MAX_LEN       = 128

In [3]:
# ── Load vocabulary ───────────────────────────────────────────────────────────
with open(VOCAB_PATH, 'rb') as f:
    vocab = pickle.load(f)
print(f'Vocab size: {len(vocab):,}')

Vocab size: 146,470


In [4]:
# ── Load model (auto-detects architecture from checkpoint) ────────────────────
def detect_arch(sd):
    d_model  = sd['encoder.embedding.token_emb.weight'].shape[1]
    d_ff     = sd['encoder.layers.0.ffn.fc1.weight'].shape[0]
    n_layers = sum(1 for k in sd if k.startswith('encoder.layers.') and k.endswith('.norm1.weight'))
    n_heads  = max(1, d_model // 64)
    return dict(d_model=d_model, n_layers=n_layers, n_heads=n_heads, d_ff=d_ff)

state = torch.load(MODEL_PATH, map_location=DEVICE)
arch  = detect_arch(state)
print('Detected:', arch)

model = Transformer(
    vocab_size=len(vocab),
    d_model=arch['d_model'],
    n_layers=arch['n_layers'],
    n_heads=arch['n_heads'],
    d_ff=arch['d_ff'],
    max_len=MAX_LEN,
    dropout=0.0,
    pooling='mean',
).to(DEVICE)

model.load_state_dict(state)
model.eval()
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

Detected: {'d_model': 384, 'n_layers': 6, 'n_heads': 6, 'd_ff': 1536}
Params: 66,891,264


In [5]:
# ── Validation set ────────────────────────────────────────────────────────────
print('VAL ->', VAL_PATH)
val_metrics = evaluate(
    model=model, csv_path=VAL_PATH, vocab=vocab,
    max_len=MAX_LEN, device=DEVICE,
    pos_threshold=POS_THRESHOLD, batch_size=BATCH_SIZE,
)
print({k: f'{v:.4f}' for k, v in val_metrics.items()})

VAL -> data/sts-222/stsb_validation.csv
    Encoding 181018 sentences ...


KeyboardInterrupt: 

In [ ]:
# ── Test set ──────────────────────────────────────────────────────────────────
print('TEST ->', TEST_PATH)
test_metrics = evaluate(
    model=model, csv_path=TEST_PATH, vocab=vocab,
    max_len=MAX_LEN, device=DEVICE,
    pos_threshold=POS_THRESHOLD, batch_size=BATCH_SIZE,
)
print({k: f'{v:.4f}' for k, v in test_metrics.items()})

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
df = pd.DataFrame([val_metrics, test_metrics], index=['Validation', 'Test'])
df.columns = ['R@1', 'R@5', 'R@10', 'MRR', 'Spearman']
print(f'Model: {MODEL_PATH}   arch: {arch}')
print()
print(df.to_string(float_format='{:.4f}'.format))

## (Optional) Compare with another checkpoint
Change `ALT_PATH` to any `.pt` file (e.g. from `compare_runs/`).

In [ ]:
ALT_PATH = 'compare_runs/cosent/best_model.pt'   # <-- change this

if os.path.exists(ALT_PATH):
    alt_state = torch.load(ALT_PATH, map_location=DEVICE)
    alt_arch  = detect_arch(alt_state)
    alt_model = Transformer(vocab_size=len(vocab), max_len=MAX_LEN, dropout=0.0, pooling='mean', **alt_arch).to(DEVICE)
    alt_model.load_state_dict(alt_state)
    alt_model.eval()

    alt_val  = evaluate(model=alt_model, csv_path=VAL_PATH,  vocab=vocab, max_len=MAX_LEN, device=DEVICE, pos_threshold=POS_THRESHOLD, batch_size=BATCH_SIZE)
    alt_test = evaluate(model=alt_model, csv_path=TEST_PATH, vocab=vocab, max_len=MAX_LEN, device=DEVICE, pos_threshold=POS_THRESHOLD, batch_size=BATCH_SIZE)

    alt_df = pd.DataFrame([alt_val, alt_test], index=['Validation', 'Test'])
    alt_df.columns = ['R@1', 'R@5', 'R@10', 'MRR', 'Spearman']
    print(f'Alt model: {ALT_PATH}   arch: {alt_arch}')
    print()
    print(alt_df.to_string(float_format='{:.4f}'.format))
else:
    print(f'Not found: {ALT_PATH}')